In [159]:
import pandas as pd

In [160]:
df = pd.read_json("1-1.여성의류(196).json")
df.head()

,Index,RawText,Source,Domain,MainCategory,ProductName,Syllable,Word,GeneralPolarity,Aspects
0,1024338,안녕하세요 이웃님들 반갑습니다. 요즘 날씨가 많이 쌀쌀해졌죠? 요즘 계절에 입으면 ...,SNS,패션,여성의류,OO 경량 다운 자켓,513,121,1,"[{'Aspect': '디자인', 'SentimentText': '딱 기본 스타일인..."
1,1024477,드디어 겨울이 찾아왔네요. 이제부터 슬슬 겨울 패딩 장만하셔야지요? 패딩 소개해 드...,SNS,패션,여성의류,OO 아** 구스코트,464,105,1,"[{'Aspect': '사이즈', 'SentimentText': '저는 블랙90 사..."
2,1025044,오늘도 정말 춥네요... 롱패딩 찾고 계신 분을 위한 후기 공유합니다. 키 158...,SNS,패션,여성의류,OO 아** 구스코트,314,78,1,"[{'Aspect': '사이즈', 'SentimentText': '키 158센티로에..."
3,1025046,이웃님들 오늘도 안녕하신가요? 오늘은 따끈한 신상 패딩 후기 올려봅니다~~ 겨울이...,SNS,패션,여성의류,OO 아** 구스코트,307,74,1,"[{'Aspect': '색상', 'SentimentText': '흰색 패딩이 너무나..."
4,1025071,OOO 구스로 소문난 OO의 롱패딩~ 한번 구경 가봐요. 일단 보는 순간 고급스럽...,SNS,패션,여성의류,OO 아** 구스코트,279,68,1,"[{'Aspect': '소재', 'SentimentText': ' 일단 보는 순간 ..."


In [161]:
len(df)

123

## 문제
- 데이터프레임에서 'Aspects' 컬럼에 데이터들을 이용하여 분류 모델을 생성하려 한다.
- SentimentText 텍스트를 이용하여 'Aspect', 'SentimentPolarity'의 값을 예측하는 모델을 생성
    1. df에서 'Aspects' 데이터를 추출
    2. SentimentText데이터를 문자형으로 이루어져있으니 학습에 대한 데이터 형태로 변환(문자의 데이터를 숫자형 데이터) -> 토큰화(okt), 벡터화(TF-IDF)
    3. 종속 변수는 'Aspect', 'SentimentPolarity'
    4. 분류 모델(LinearSVC) random_state만 42로 고정
    5. 테스트데이터를 이용하여 분류가 잘되고 있는가? 정확도만 확인
        - 벡터화, 모델링 파이프라인으로 연결해서 사용

In [162]:
from sklearn.svm import LinearSVC
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score
from konlpy.tag import Okt
from sklearn.pipeline import Pipeline

In [163]:
test_text = [
    '색상이 마음에 든다',
    '설명에 비해 옷이 두껍진 않다',
    '길이가 너무 길지도 않고 짧지도 않다'
]

In [164]:
data = df['Aspects']

In [165]:
len(data)

123

In [166]:
data[0][0]

{'Aspect': '디자인',
 'SentimentText': '딱 기본 스타일인데 또 입은 거 보면 깔끔하게 저렴해보이지 않는 디자인이라서',
 'SentimentWord': '11',
 'SentimentPolarity': '1'}

In [167]:
type(data)

pandas.core.series.Series

In [168]:
pd.DataFrame(df['Aspects'].sum())

,Aspect,SentimentText,SentimentWord,SentimentPolarity
0,디자인,딱 기본 스타일인데 또 입은 거 보면 깔끔하게 저렴해보이지 않는 디자인이라서,11,1
1,두께,이것만 입기엔 얇지만,3,-1
2,기능,초겨울까지는 운동 갈때 안에 얇은 기능성 반팔 입고 요것만 입어도 꽤 따뜻해요.,12,1
3,색상,색상도 디자인도 무난해서,3,0
4,디자인,디자인도 무난해서,2,0
...,...,...,...,...
912,길이,길이감도 너무 짧거나 애매한 길이가 아니고 적당합니다.,7,1
913,활용성,트레이닝 세트나 데님 스커트 후드 원피스 등에도 잘 어울립니다.,9,1
914,디자인,전체적으로 고급스러움이 잘 녹아있는 디자인으로 되어 있고,7,1
915,품질,퀄리티도 넘 휼륭합니다.,3,1


In [169]:
X = []
Y = []

for item in data:
    for aspect in item:
        pair = []
        X.append(aspect['SentimentText'])
        pair.append(aspect['Aspect'])
        pair.append(aspect['SentimentPolarity'])
        Y.append(pair)


In [170]:
X = pd.DataFrame(X, columns=['SentimentText'])
Y = pd.DataFrame(Y, columns=['Aspect', 'SentimentPolarity'])

In [171]:
X

,SentimentText
0,딱 기본 스타일인데 또 입은 거 보면 깔끔하게 저렴해보이지 않는 디자인이라서
1,이것만 입기엔 얇지만
2,초겨울까지는 운동 갈때 안에 얇은 기능성 반팔 입고 요것만 입어도 꽤 따뜻해요.
3,색상도 디자인도 무난해서
4,디자인도 무난해서
...,...
912,길이감도 너무 짧거나 애매한 길이가 아니고 적당합니다.
913,트레이닝 세트나 데님 스커트 후드 원피스 등에도 잘 어울립니다.
914,전체적으로 고급스러움이 잘 녹아있는 디자인으로 되어 있고
915,퀄리티도 넘 휼륭합니다.


In [172]:
Y

,Aspect,SentimentPolarity
0,디자인,1
1,두께,-1
2,기능,1
3,색상,0
4,디자인,0
...,...,...
912,길이,1
913,활용성,1
914,디자인,1
915,품질,1


In [173]:
X.iloc[:, :2] = X.iloc[:, :2].map(lambda x : x.strip())
Y.iloc[:, :2] = Y.iloc[:, :2].map(lambda x : x.strip())

In [174]:
# 두 개의 종속변수를 합친 새로운 종속변수 생성
Y_combined = Y['Aspect'] + '_' + Y['SentimentPolarity']

In [175]:
from sklearn.preprocessing import LabelEncoder

In [176]:
le = LabelEncoder()
Y['Aspect'] = le.fit_transform(Y['Aspect'])

In [177]:
okt = Okt()
def tokenize(text):
    return okt.morphs(text)

vectorizer = TfidfVectorizer(tokenizer=tokenize,
                            lowercase=False,
                            ngram_range=(1, 2))

In [206]:
# 두 개의 종속 변수를 예측하는 모델 생성
svc1 = LinearSVC(random_state=42)
svc2 = LinearSVC(random_state=42)
svc3 = LinearSVC(random_state=42)

In [207]:
pipe1 = Pipeline([
    ('vector', vectorizer),
    ('clf', svc1)
])

pipe2 = Pipeline([
    ('vector', vectorizer),
    ('clf', svc2)
])

pipe3 = Pipeline([
    ('vector', vectorizer),
    ('clf', svc3)
])



In [208]:
pipe1.fit(X['SentimentText'], Y['Aspect'])

c:\Users\abohv\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\feature_extraction\text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


,steps,"[('vector', ...), ('clf', ...)]"
,transform_input,None
,memory,None
,verbose,False
,input,'content'
,encoding,'utf-8'
,decode_error,'strict'
,strip_accents,None
,lowercase,False
,preprocessor,None
,tokenizer,<function tok...002F2CA2E3B00>


In [209]:
pipe2.fit(X['SentimentText'], Y['SentimentPolarity']) 

c:\Users\abohv\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\feature_extraction\text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


,steps,"[('vector', ...), ('clf', ...)]"
,transform_input,None
,memory,None
,verbose,False
,input,'content'
,encoding,'utf-8'
,decode_error,'strict'
,strip_accents,None
,lowercase,False
,preprocessor,None
,tokenizer,<function tok...002F2CA2E3B00>


In [210]:
pipe3.fit(X['SentimentText'], Y_combined)

c:\Users\abohv\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\feature_extraction\text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


,steps,"[('vector', ...), ('clf', ...)]"
,transform_input,None
,memory,None
,verbose,False
,input,'content'
,encoding,'utf-8'
,decode_error,'strict'
,strip_accents,None
,lowercase,False
,preprocessor,None
,tokenizer,<function tok...002F2CA2E3B00>


In [211]:
pred1 = pipe1.predict(test_text)
pred2 = pipe2.predict(test_text)
for i in range(len(test_text)):
    print(f"Test: {test_text[i]} \n 예측된 Aspect: {le.inverse_transform(pred1)[i]} \n 예측된 SentimentPolarity: {pred2[i]}  \n\n")

Test: 색상이 마음에 든다 
 예측된 Aspect: 색상 
 예측된 SentimentPolarity: 1  


Test: 설명에 비해 옷이 두껍진 않다 
 예측된 Aspect: 디자인 
 예측된 SentimentPolarity: 1  


Test: 길이가 너무 길지도 않고 짧지도 않다 
 예측된 Aspect: 길이 
 예측된 SentimentPolarity: 1  




In [212]:
pred3 = pipe3.predict(test_text)
for i in range(len(test_text)):
    print(f"Test: {test_text[i]} \n 예측된 Aspect_SentimentPolarity: {pred3[i]} \n\n")

Test: 색상이 마음에 든다 
 예측된 Aspect_SentimentPolarity: 색상_1 


Test: 설명에 비해 옷이 두껍진 않다 
 예측된 Aspect_SentimentPolarity: 소재_1 


Test: 길이가 너무 길지도 않고 짧지도 않다 
 예측된 Aspect_SentimentPolarity: 길이_1 




In [213]:
# 종속 변수가 2개인 경우 일반적으로 사용하는 객체
from sklearn.multioutput import MultiOutputClassifier

In [214]:
# 종속 변수의 크기가 (2000, 2)
# 첫번째 종속의 데이터를 이용하여 fit -> predict()
# 두번째 종속으 ㅣ데이터를 이용하여 fit() -> predict()
# 위의 2번째 직업을 병렬로 처리

In [217]:
svc = LinearSVC()

In [218]:
# 분류 모델을 생성
svc = LinearSVC(random_state=42)
# 멀티 아웃 모델을 생성
multi_model = MultiOutputClassifier(svc)
# 파이프라인 생성
pipe_multi = Pipeline(
    [
        ('vector', vectorizer),
        ('clf', multi_model)
    ]
)

In [226]:
Y['SentimentPolarity'] = Y['SentimentPolarity'].astype(int)

In [233]:
x = X['SentimentText'].values
y = Y[['Aspect', 'SentimentPolarity']].values

In [234]:
# 멀티 모델 종속은 2차원 그대로 사용
pipe_multi.fit(x, y)

c:\Users\abohv\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\feature_extraction\text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


,steps,"[('vector', ...), ('clf', ...)]"
,transform_input,None
,memory,None
,verbose,False
,input,'content'
,encoding,'utf-8'
,decode_error,'strict'
,strip_accents,None
,lowercase,False
,preprocessor,None
,tokenizer,<function tok...002F2CA2E3B00>


In [235]:
pred_multi = pipe_multi.predict(test_text)

In [236]:
pred_multi

array([[8, 1],
       [4, 1],
       [2, 1]])

In [237]:
# 문단인 장문의 데이터에서 문장별로 나눠주기
from konlpy.tag import Kkma

In [238]:
text = df.loc[0, 'RawText']

In [239]:
kkma = Kkma()

In [243]:
texts = kkma.sentences(text)

In [244]:
pred = pipe_multi.predict(texts)